# Day 2 — Train two checkpoints on LOL-v2 Real (Kaggle T4)

This notebook trains **both** ablation variants on LOL-v2 Real:

1. `baseline` — `use_illum_prior=0` (matches the prior-disabled architecture, but trained on LOL-v2 Real).
2. `+illum`   — `use_illum_prior=1` (the new architectural variant; this is the novelty contribution).

Why two trainings: a clean method ablation needs identical training data + protocol, varying only the architecture flag. Comparing the new `+illum` model against the old eval15-trained checkpoint would confound architecture with training data.

Estimated wall time on T4: **~5–7 hours total** (2.5–3.5 hours per run × 2). Comfortably fits one Kaggle session.

**If you only have time for one training run, run Cell 7 (the `+illum` variant) first.** That is the one you actually need for the paper. If time permits afterwards, run Cell 6 (`baseline`) for the ablation row.

**Before running:** push the latest local commits (especially the `--dataset-root` flag in `train.py`).

In [ ]:
# === Cell 1: install ===
!pip install -q --upgrade pip
!pip install -q scikit-image lpips matplotlib

In [ ]:
# === Cell 2: clone the repo ===
import os, sys, shutil, subprocess

BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

with open("train.py") as f:
    has_root = "--dataset-root" in f.read()
print("--dataset-root flag present in train.py:", has_root)
if not has_root:
    raise SystemExit("Push your latest commits first; train.py on GitHub doesn't have --dataset-root yet.")

In [ ]:
# === Cell 3: discover LOL-v2 Real dataset root ===
# We need the directory that CONTAINS Real_captured/Train/Low (i.e. the parent of Real_captured).
import glob

candidates = sorted({os.path.dirname(p) for p in
                     glob.glob("/kaggle/input/**/Real_captured/Train/Low", recursive=True)})
if not candidates:
    raise SystemExit(
        "Could not find /kaggle/input/**/Real_captured/Train/Low. "
        "Did you attach the LOL-v2 dataset via the Kaggle right panel (+ Add Data)?"
    )
DATASET_ROOT = candidates[0]
print("DATASET_ROOT =", DATASET_ROOT)

# Sanity-check the four required subfolders
for sub in ("Real_captured/Train/Low", "Real_captured/Train/Normal",
            "Real_captured/Test/Low",  "Real_captured/Test/Normal"):
    p = os.path.join(DATASET_ROOT, sub)
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {sub:<35s}  {n} files")

In [ ]:
# === Cell 4: GPU sanity ===
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Accelerator -> GPU T4")

In [ ]:
# === Cell 5: training hyperparameters (edit if needed) ===
EPOCHS      = 120     # 100-150 is a reasonable range; 120 fits in ~2.5h/run on T4
CROP_SIZE   = 256     # do not go below 192 — LOL-v2 Real images are ~600x400
BATCH_SIZE  = 4       # T4 (16GB) handles batch 4 at 256-crop comfortably
VAL_EVERY   = 5       # validate + save best every N epochs (env var)

ENV_OVERRIDES = {
    "EPOCHS":     str(EPOCHS),
    "CROP_SIZE":  str(CROP_SIZE),
    "BATCH_SIZE": str(BATCH_SIZE),
    "VAL_EVERY":  str(VAL_EVERY),
}
for k, v in ENV_OVERRIDES.items():
    os.environ[k] = v

print("Training config:")
for k, v in ENV_OVERRIDES.items():
    print(f"  {k} = {v}")

In [ ]:
# === Cell 6 (OPTIONAL — run only if you have time): baseline (no illum prior) ===
# Skip this cell if you're short on time. The +illum cell below is what you NEED.
# Outputs ./checkpoints/best_lolv2real_baseline.pth  +  last_lolv2real_baseline.pth
import sys, subprocess
cmd = [
    sys.executable, "train.py",
    "--layout", "lolv2_real",
    "--dataset-root", DATASET_ROOT,
    "--use-illum-prior", "0",
    "--epochs", str(EPOCHS),
    "--crop-size", str(CROP_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", "2",
    "--tag", "lolv2real_baseline",
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 7: REQUIRED — train +illum (the headline novelty checkpoint) ===
# Outputs ./checkpoints/best_lolv2real_illum.pth  +  last_lolv2real_illum.pth
cmd = [
    sys.executable, "train.py",
    "--layout", "lolv2_real",
    "--dataset-root", DATASET_ROOT,
    "--use-illum-prior", "1",
    "--epochs", str(EPOCHS),
    "--crop-size", str(CROP_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", "2",
    "--tag", "lolv2real_illum",
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 8: collect checkpoints + train logs into /kaggle/working ===
import shutil, glob
OUT = "/kaggle/working/day2_checkpoints"
os.makedirs(OUT, exist_ok=True)

for src in glob.glob("./checkpoints/best_lolv2real*.pth"):
    shutil.copy2(src, OUT)
    print("copied", src)
for src in glob.glob("./checkpoints/last_lolv2real*.pth"):
    shutil.copy2(src, OUT)
    print("copied", src)
for src in glob.glob("./checkpoints/train_log_lolv2real*.csv"):
    shutil.copy2(src, OUT)
    print("copied", src)

# print val PSNR trajectory from the train logs so you can sanity-check convergence
import csv
for log in sorted(glob.glob(os.path.join(OUT, "train_log_*.csv"))):
    print(f"\n--- {log} ---")
    rows = list(csv.reader(open(log)))
    header = rows[0]
    val_psnr_idx = header.index("val_psnr")
    print("epoch  val_psnr  val_ssim")
    for r in rows[1:]:
        if r[val_psnr_idx] not in ("-1.0", ""):
            print(f"{r[0]:<5}  {float(r[val_psnr_idx]):.3f}  {r[header.index('val_ssim')]}")

out_zip = "/kaggle/working/day2_checkpoints.zip"
if os.path.exists(out_zip): os.remove(out_zip)
shutil.make_archive(out_zip.replace(".zip", ""), "zip", OUT)
print("\nWrote", out_zip)
print("Download via Kaggle right panel -> Output tab.")
print("Then: upload the .pth files as a NEW Kaggle dataset named 'lumidiff-day2-ckpts' ")
print("and attach to the Day 2 EVAL notebook.")